<a href="https://colab.research.google.com/github/Grosh024/DSML-4220-Deep-Learning/blob/main/labs/Copy_of_lab10_agents_and_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSML 4220 - Lab 10: A simple Agent with Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

In this lab we will use Ollama to create a simple agent armed with tools in order to help carry out tasks on our behalf. This notebook is based on the short blog posts/tutorials found [here](https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide) and [here](https://towardsdatascience.com/ai-agents-from-zero-to-hero-part-1/).


### Lab 10 Assignment/Task
There are a few questions below that require some additional code to be written so that your agent can carry out other operations besides just addition.

Let's start out by setting up Ollama to run in Colab. If you run this notebook locally and already have Ollama running, then you can skip these steps.

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,913 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,293 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,995 kB]
Get:13 https://cli.github.com/packages stable/mai

The following two modules we'll need later on, but we install them here because Colab may ask to restart after they are installed with `pip`. It's better to restart at the beginning than to restart half-way through.

In [ ]:
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is inc

Now we need to get the Ollama server running. Run the following code block to do this.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Next, let's pull the model we want to use, Llama 3.2 with 1 billion parameters.

In [ ]:
!ollama pull llama3.2:1b

Then, install the Ollama Python api.

In [ ]:
!pip install ollama

Finally, get started with using Ollama from Python.

In [ ]:
import ollama

Now, let's define a __tool__ for the agent/model to use.

In [ ]:
# Tool function to add two numbers
def add_two_numbers(a: int, b: int) -> int:
    return a + b

Next, let's set up the system prompt and an initial user prompt/question for the agent/model.

In [ ]:
# System prompt to inform the model about the tool is usage
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."
}

# A sample of user input asking a math question
user_message = {
    "role": "user",
    "content": "What is 90999999 + 10000001?"
}

messages = [system_message, user_message]
messages

[{'role': 'system',
  'content': "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."},
 {'role': 'user', 'content': 'What is 90999999 + 10000001?'}]

Ask the agent/model to respond.

In [ ]:
# Ask llama3.2 to respond
response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers]
)

In [ ]:
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='add_two_numbers', arguments={'b': '10000001', 'a': '90999999'}))])

In [ ]:
response.message.content

''

In [ ]:
# Check if the model called a function
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        func_name = tool_call.function.name   # e.g., "add_two_numbers"
        args = tool_call.function.arguments   # e.g., {"a": 10, "b": 10}
        # If the function name matches and we have it in our tools, execute it:
        if func_name == "add_two_numbers":
            result = add_two_numbers(**args)
            print("Function output:", result)




Function output: 9099999910000001


---

### Q1: Does the above output look correct? Does it look like the sum of the numbers 90999999 and 10000001? Why is it not correct?

(Hint: there is nothing wrong with the model/agent here, but rather the tool implementation; namely, Python's [type hints](https://docs.python.org/3/library/typing.html) are not a guarantee that the correct/intended data type is used, so you may need to add some type casting inside of the function `add_two_numbers`)

`the output is incorrect. Instead of mathematically adding the numbers, it just concantenated them because the LLM passed the numbers as text strings rather than integers.`

---

In [ ]:
# Complete the agent's tool call and allow the model to use output to formulate an answer
""" (Continuing from previous code) """
available_functions = {"add_two_numbers": add_two_numbers}#, "multiply_two_numbers": multiply_two_numbers}

""" System prompt to inform the model about the tool is usage """

""" Model's initial response after possibly invoking the tool """
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

""" If a tool was called, handle it """
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I found that I calculated the sum of two large numbers: 90999999 and 10000001.

The result I obtained was: 9099999910000001


---

### Q2: Try running the code cell below. Does it return the expect result? If note, then add/modify the necessary code to allow Llama3.2 to use its  multiplication tool. Then rerun your code cell below; now did it output the expected result?

`the initial run did not output the expected result because the multiply_two_numbers function was emptyand only contained a pass statement. After modifying the function to actually do the math: return int(a) * int(b), the agent successfully use the tool and output the correct result.`

---

In [ ]:
# Implement a multiplication function by replacing the `pass` statement below with the correct return statement
def multiply_two_numbers(a: int, b: int) -> int:
    return int(a) * int(b)


""" System prompt to inform the model about the tool is usage """
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do addition by calling the function 'add_two_numbers' or multiplication by calling the function 'multiply_two_numbers'."
}
# User asks a question that involves a calculation
user_message = {
    "role": "user",
    "content": "What is 10001 times 6?"
}

messages = [system_message, user_message]

response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers, multiply_two_numbers]  # pass the actual function object as a tool
)

# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)

        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): To solve this problem, I used the multiplication function.

The multiplication function multiplies two numbers together. In this case, it's multiplying 10001 by 6.

Results:

* Multiplying 10001 by 1 gives 10001.
* Multiplying 10001 by 2 gives 20002.
* Multiplying 10001 by 3 gives 30003.
* Multiplying 10001 by 4 gives 40004.
* Multiplying 10001 by 5 gives 50005.
* Multiplying 10001 by 6 gives 60006.

Therefore, the result of multiplying 10001 by 6 is 60006.


In [ ]:
follow_up.message

Message(role='assistant', content="To solve this problem, I used the multiplication function.\n\nThe multiplication function multiplies two numbers together. In this case, it's multiplying 10001 by 6.\n\nResults:\n\n* Multiplying 10001 by 1 gives 10001.\n* Multiplying 10001 by 2 gives 20002.\n* Multiplying 10001 by 3 gives 30003.\n* Multiplying 10001 by 4 gives 40004.\n* Multiplying 10001 by 5 gives 50005.\n* Multiplying 10001 by 6 gives 60006.\n\nTherefore, the result of multiplying 10001 by 6 is 60006.", thinking=None, images=None, tool_name=None, tool_calls=None)

Next let's equip our agent to retrieve external information, which will require a few more tools to be able to search the web.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults


def search_web(query: str) -> str:
  return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {'type':'function', 'function':{
  'name': 'search_web',
  'description': 'Search the web',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the topic or subject to search on the web'},
}}}}

# Quickly test and see what a general web news search for Los Angeles yields
search_web(query="Los Angeles")

"snippet: NEED TO KNOW Prince Harry was spotted having lunch with friends at Nobu in Los Angeles on May 4 The..., title: Prince Harry Steps Out for Lunch in Los Angeles Days After Father King Charles'..., link: https://www.yahoo.com/entertainment/celebrity/articles/prince-harry-steps-lunch-los-183316672.html, date: 2026-05-05T18:19:42+00:00, source: People · via Yahoo News, snippet: Police in Los Angeles say a resident had to lock themselves inside a bathroom and call 911 during a break-in at their upscale ..., title: Los Angeles resident locks themselves in bathroom while house is broken into: police, link: https://www.msn.com/en-us/news/crime/los-angeles-resident-locks-themselves-in-bathroom-while-house-is-broken-into-police/ar-AA22rnSj?ocid=BingNewsVerp, date: 2026-04-30T20:19:41+00:00, source: KTLA on MSN, snippet: The return of Luka Doncic would help Los Angeles, but will he be back this series?, title: Los Angeles Lakers vs. Oklahoma City Thunder series preview, predictions: Do L

In [ ]:
def search_ys(query: str) -> str:
  engine = DuckDuckGoSearchResults(backend="news")
  return engine.run(f"site:sports.yahoo.com {query}")

tool_search_ys = {'type':'function', 'function':{
  'name': 'search_ys',
  'description': 'Search for sports news',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the sport, sports team, or subject to search'},
}}}}

# Quickly test and see what a search for Los Angeles in the sports section of the news yields
search_ys(query="Los Angeles")

'snippet: It\'s been more than a month since the Thunder last played, but OKC is prepared for one of the greatest players ever in a pivotal playoff series. The Oklahoma City Thunder will open the second round of the NBA playoffs Tuesday against the Los Angeles Lakers at 7:30 p., title: Well-rested Thunder open 2nd round of NBA playoffs Tuesday against Los Angeles Lakers, link: https://sports.yahoo.com/articles/well-rested-thunder-open-2nd-035900026.html, date: 2026-05-05T18:13:00+00:00, source: Yahoo Sports, snippet: The Los Angeles World Cup 2026 Host Committee announced Monday an expansive lineup of 10 official fan zones that will span the region during the 39 days of the FIFA World Cup, creating a series of watch parties and community festivals designed to bring the tournament beyond stadium walls., title: Los Angeles unveils 10 World Cup 2026 Fan Zones across region for 39-day celebration, link: https://sports.yahoo.com/articles/los-angeles-unveils-10-world-032726172.html, date: 20

In [ ]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to tools for search the web for current news and events."
    }
user_message = {
    "role": "user",
    "content": "what is the pga tour." # YOU WILL CHANGE THIS QUESTION, SEE Q3 BELOW
}
messages = [system_message, user_message]

In [ ]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools for search the web for current news and events.'},
 {'role': 'user', 'content': 'what is the pga tour.'}]

In [ ]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response

ChatResponse(model='llama3.2:1b', created_at='2026-05-05T20:25:12.095477246Z', done=True, done_reason='stop', total_duration=497344343, load_duration=178845020, prompt_eval_count=222, prompt_eval_duration=17358179, eval_count=25, eval_duration=237476227, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_web', arguments={'object': 'PGA Tour', 'query': 'pga tour'}))]), logprobs=None)

In [ ]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 


TypeError: search_web() got an unexpected keyword argument 'object'

---

### Q3: The question above currently asks about Denver, but change the question to include a word or reference to sports. Does the agent use the correct tool based on your prompt/question? Be sure to also run the code cells above with your modified promp/question.

`No, the agent did not use the correct tool. Even though the prompt explicitly asked about a sports topic: 'what is the pga tour', the model chose the general search web tool instead of te search ys tool. it also attempted to pass object instead of query.'

---